In [1]:
import pandas as pd
import numpy as np

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

In [2]:
features = pd.read_csv("../data/processed/provider_features.csv")

print(features.shape)

(1296739, 92)


In [3]:
model_features = [
    "payment_per_beneficiary",
    "services_per_beneficiary",
    "charge_to_payment_ratio",
    "allowed_to_charge_ratio",
    "standardized_payment_ratio",
    "payment_vs_state_avg",
    "payment_vs_specialty_avg",
    "tot_mdcr_pymt_amt",
    "tot_srvcs",
    "tot_benes"
]

X = features[model_features]

In [4]:
X = X.fillna(X.median())

In [5]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

In [6]:
model = IsolationForest(
    n_estimators=200,
    contamination=0.02,
    random_state=42
)

model.fit(X_scaled)

print("Isolation Forest trained successfully.")

Isolation Forest trained successfully.


In [7]:
features["anomaly"] = model.predict(X_scaled)

In [8]:
scores = -model.decision_function(X_scaled)

scores = (
    (scores - scores.min()) /
    (scores.max() - scores.min())
) * 100

features["fraud_risk_score"] = scores

In [9]:
risk_columns = [
    "rndrng_npi",
    "rndrng_prvdr_last_org_name",
    "rndrng_prvdr_first_name",
    "rndrng_prvdr_type",
    "rndrng_prvdr_state_abrvtn",
    "fraud_risk_score"
]

high_risk = (
    features[risk_columns]
    .sort_values("fraud_risk_score", ascending=False)
)

high_risk.head(20)

,rndrng_npi,rndrng_prvdr_last_org_name,rndrng_prvdr_first_name,rndrng_prvdr_type,rndrng_prvdr_state_abrvtn,fraud_risk_score
1238979,1952581027,Hardesty,Brandon,Internal Medicine,IN,100.000000
850154,1659532927,Hedrick,David,Internal Medicine,IN,100.000000
37602,1023678240,Washington Institute For Coagulation,Unknown,Pharmacy,WA,99.611729
1193596,1922019561,Cascade Hemophilia Consortium,Unknown,Pharmacy,MI,99.556337
1185072,1912332289,Henning,Andrew,Nurse Practitioner,OR,99.445609
1139410,1871881573,Ethical Factor Rx Llc,Unknown,Pharmacy,PA,99.445609
499709,1386627966,Orthopaedic Hospital,Unknown,Pharmacy,CA,99.390273
762587,1588766968,Regents Of The University Of Colorado,Unknown,Pharmacy,CO,99.169116
713351,1548904360,Barton,Sherri,Nurse Practitioner,FL,99.003444
904839,1699778563,Hemophilia Of Georgia,Unknown,Pharmacy,GA,98.948258


In [10]:
features.to_csv(
    "../data/processed/provider_risk_scores.csv",
    index=False
)

print("Fraud risk scores saved successfully!")

Fraud risk scores saved successfully!


In [13]:
def risk_category(score):
    if score >= 80:
        return "High Risk"
    elif score >= 50:
        return "Medium Risk"
    else:
        return "Low Risk"

features["risk_category"] = features["fraud_risk_score"].apply(risk_category)

features["risk_category"].value_counts()

risk_category
Low Risk       1268774
Medium Risk      22345
High Risk         5620
Name: count, dtype: int64

In [14]:
import matplotlib.pyplot as plt
import os

os.makedirs("../Images", exist_ok=True)

risk_counts = features["risk_category"].value_counts()

plt.figure(figsize=(6, 4))
risk_counts.plot(kind="bar")
plt.title("Risk Category Distribution")
plt.xlabel("Risk Category")
plt.ylabel("Provider Count")
plt.tight_layout()

plt.savefig("../Images/risk_distribution.png", dpi=300, bbox_inches="tight")
plt.close()

print("Risk distribution image saved.")

Risk distribution image saved.
